In [ ]:
import torch
import torchvision.ops as ops

# ── Intersection over Union ───────────────────────────────────────────
def iou(box1, box2):
    """
    Compute IoU between two boxes.
    Boxes: (x1, y1, x2, y2) format, where (x1,y1) is top-left.
    """
    # Intersection
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)

    # Union
    a1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    a2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    union = a1 + a2 - inter
    return inter / (union + 1e-8)

pred_box  = [50, 50, 200, 200]
gt_box    = [80, 80, 220, 220]
print(f'IoU: {iou(pred_box, gt_box):.3f}')   # >0.5 = good match

# ── Non-Maximum Suppression (NMS) ─────────────────────────────────────
# After detection, many overlapping boxes are predicted for the same object.
# NMS keeps only the highest-scoring box from each cluster of overlaps.
boxes  = torch.tensor([[100,100,200,200],[110,110,210,210],[300,300,400,400]],
                       dtype=torch.float32)
scores = torch.tensor([0.9, 0.75, 0.85])

# torchvision.ops.nms returns indices of kept boxes
kept_idx = ops.nms(boxes, scores, iou_threshold=0.5)
print(f'After NMS: kept boxes at indices {kept_idx.tolist()}')
# boxes[0] and boxes[1] overlap heavily → only boxes[0] (score 0.9) is kept
# boxes[2] does not overlap → also kept

# ── torchvision detection metrics ────────────────────────────────────
# For mAP calculation, use torchmetrics or torchvision built-ins:
# from torchmetrics.detection.mean_ap import MeanAveragePrecision
# metric = MeanAveragePrecision(iou_thresholds=[0.5, 0.75])


In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

# ── Converting a classifier to a Fully Convolutional Network ─────────

# Standard classifier: only works on fixed input size
class FixedSizeClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),   # 128 → 64
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),   # 64 → 32
            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),   # 32 → 16
        )
        # FC layer: ONLY works when backbone output is exactly 16×16
        self.classifier = nn.Linear(256 * 16 * 16, 1000)  # BROKEN for other sizes

    def forward(self, x):
        x = self.backbone(x).flatten(1)   # breaks if input != 128×128
        return self.classifier(x)

# FCN: works on ANY input size
class FullyConvolutionalNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        # Replace FC with 1×1 conv: works on any spatial size!
        self.classifier = nn.Conv2d(256, 1000, kernel_size=1)  # 1×1 conv

    def forward(self, x):
        feats = self.backbone(x)          # (B, 256, H/8, W/8)
        scores = self.classifier(feats)   # (B, 1000, H/8, W/8) — spatial grid!
        return scores

fcn = FullyConvolutionalNetwork()

# Works on any input size:
for h, w in [(128, 128), (256, 256), (256, 384)]:
    x = torch.randn(1, 3, h, w)
    out = fcn(x)
    print(f'Input {h}×{w} → output {tuple(out.shape[2:])} spatial cells')
# 128×128 → (16,16) cells  — each sees a 128/16 = 8px receptive field
# 256×256 → (32,32) cells
# 256×384 → (32,48) cells  — efficient sliding window in one pass!

# To use as a plain classifier (for training), add global average pooling:
gap_out = fcn(torch.randn(1, 3, 128, 128)).mean(dim=[2,3])  # (1, 1000)
print(f'After GAP: {gap_out.shape}')   # single class score vector


In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

# ── Classification + Localisation with dual heads ────────────────────
class ClassificationAndLocalisation(nn.Module):
    def __init__(self, num_classes=20, pretrained=True):
        super().__init__()
        backbone = models.resnet50(
            weights=models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)

        # Shared feature extractor (everything except the final FC)
        self.features   = nn.Sequential(*list(backbone.children())[:-1])
        feat_dim        = 2048  # ResNet50 output dimension

        # Classification head: C class scores
        self.cls_head   = nn.Linear(feat_dim, num_classes)

        # Regression head: 4 box coordinates (class-agnostic)
        # Outputs: (cx, cy, w, h) as fractions of image size
        self.bbox_head  = nn.Linear(feat_dim, 4)

    def forward(self, x):
        feats   = self.features(x).flatten(1)  # (B, 2048)
        cls_out = self.cls_head(feats)          # (B, num_classes) — logits
        box_out = self.bbox_head(feats).sigmoid()  # (B, 4) in [0,1]
        return cls_out, box_out

def localisation_loss(cls_pred, cls_target, box_pred, box_target,
                      cls_weight=1.0, box_weight=1.0):
    """
    Multi-task loss: classification (cross-entropy) + localisation (smooth L1).
    Smooth L1 is less sensitive to outliers than MSE.
    """
    cls_loss = nn.CrossEntropyLoss()(cls_pred, cls_target)
    # Smooth L1 = 0.5*x^2 for |x|<1, else |x|-0.5  (Huber loss)
    box_loss = nn.SmoothL1Loss()(box_pred, box_target)
    return cls_weight * cls_loss + box_weight * box_loss

# ── Training step ─────────────────────────────────────────────────────
model = ClassificationAndLocalisation(num_classes=20)
opt   = torch.optim.Adam(model.parameters(), lr=1e-4)

images   = torch.randn(4, 3, 224, 224)
cls_gt   = torch.randint(0, 20, (4,))
box_gt   = torch.rand(4, 4)   # (cx, cy, w, h) normalised to [0,1]

opt.zero_grad()
cls_pred, box_pred = model(images)
loss = localisation_loss(cls_pred, cls_gt, box_pred, box_gt)
loss.backward()
opt.step()
print(f'Loss: {loss.item():.4f}  (cls + bbox regression)')


In [ ]:
import torch
import torch.nn.functional as F

# ── RoI Pooling from scratch ──────────────────────────────────────────
# Goal: given a variable-size region proposal on the shared feature map,
# always produce a FIXED output size — e.g. (C, 7, 7).

def roi_pool(feature_map, proposal, output_size=(2, 2)):
    """
    Single-proposal RoI Pooling.
    feature_map: (C, H, W)
    proposal:    (x1, y1, x2, y2) in feature-map pixel coords
    output_size: (out_h, out_w) fixed output grid
    Returns:     (C, out_h, out_w)
    """
    C, H, W = feature_map.shape
    out_h, out_w = output_size
    x1, y1, x2, y2 = proposal

    # Size of each bin in the output grid
    bin_h = (y2 - y1) / out_h
    bin_w = (x2 - x1) / out_w

    output = torch.zeros(C, out_h, out_w)

    for i in range(out_h):       # row of output grid
        for j in range(out_w):   # column of output grid
            # Pixel range for this bin (integer boundaries)
            r0 = max(int(y1 + i * bin_h), 0)
            r1 = min(int(y1 + (i+1) * bin_h), H)
            c0 = max(int(x1 + j * bin_w), 0)
            c1 = min(int(x1 + (j+1) * bin_w), W)

            if r1 > r0 and c1 > c0:
                # Max-pool the bin across all channels
                output[:, i, j] = feature_map[:, r0:r1, c0:c1].flatten(1).max(1).values

    return output

# ── Two proposals of different sizes → same output shape ─────────────
torch.manual_seed(0)
fmap = torch.randn(4, 8, 8)   # 4-channel 8x8 feature map

# Large proposal: covers most of the feature map
out_large = roi_pool(fmap, proposal=(0, 0, 8, 8), output_size=(2, 2))
# Small proposal: top-left corner only
out_small = roi_pool(fmap, proposal=(1, 1, 4, 4), output_size=(2, 2))

print(f"Feature map: {tuple(fmap.shape)}")
print(f"Large proposal (0,0,8,8) → {tuple(out_large.shape)}")
print(f"Small proposal (1,1,4,4) → {tuple(out_small.shape)}")
# Both produce (4, 2, 2) regardless of proposal size — this is the point!

# ── torchvision built-in (batched, fast) ─────────────────────────────
import torchvision.ops as ops

# boxes: (N, 5) — each row is [batch_idx, x1, y1, x2, y2]
fmap_batch = fmap.unsqueeze(0)   # (1, 4, 8, 8)
boxes = torch.tensor([[0, 0., 0., 8., 8.],
                       [0, 1., 1., 4., 4.]], dtype=torch.float32)

pooled = ops.roi_pool(fmap_batch, boxes, output_size=(2,2), spatial_scale=1.0)
print(f"torchvision roi_pool: {tuple(pooled.shape)}")
# (2, 4, 2, 2) — 2 proposals, each (4, 2, 2)

# spatial_scale converts image coordinates to feature-map coordinates.
# If the backbone downsamples by stride 16, use spatial_scale=1/16.
# A box at image pixels (160,160,320,320) becomes (10,10,20,20) on the feature map.


In [ ]:
import torch
import torchvision
from torchvision.models.detection import (fasterrcnn_resnet50_fpn,
                                           FasterRCNN_ResNet50_FPN_Weights)

# ── Using a pretrained Faster R-CNN ──────────────────────────────────
model = fasterrcnn_resnet50_fpn(
    weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1)

# ── Inference mode ────────────────────────────────────────────────────
model.eval()
images = [torch.rand(3, 800, 600), torch.rand(3, 600, 800)]

with torch.no_grad():
    predictions = model(images)

for i, pred in enumerate(predictions):
    print(f'Image {i}: {len(pred["boxes"])} detections')
    print(f'  boxes:  {pred["boxes"].shape}')   # (N, 4) in xyxy format
    print(f'  labels: {pred["labels"].shape}')  # (N,)   COCO class ids
    print(f'  scores: {pred["scores"].shape}')  # (N,)   confidence

# ── Training mode: pass image list + target dicts ────────────────────
model.train()
targets = [
    {'boxes':  torch.tensor([[10, 20, 100, 200], [50, 60, 150, 250]],
                             dtype=torch.float32),
     'labels': torch.tensor([1, 2])},   # COCO class ids
    {'boxes':  torch.tensor([[30, 40, 200, 300]], dtype=torch.float32),
     'labels': torch.tensor([3])},
]

loss_dict = model(images, targets)
# loss_dict contains four losses:
# 'loss_classifier', 'loss_box_reg',    ← detection head losses
# 'loss_objectness', 'loss_rpn_box_reg'  ← RPN losses
total_loss = sum(loss_dict.values())
print('\nLoss components:')
for k, v in loss_dict.items():
    print(f'  {k}: {v.item():.4f}')
print(f'Total loss: {total_loss.item():.4f}')

# ── Fine-tuning for a custom dataset ─────────────────────────────────
# 1. Change the number of output classes
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

model = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1)
in_features = model.roi_heads.box_predictor.cls_score.in_features
num_classes = 5  # your dataset (including background)
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
print(f'Custom detector: {num_classes} classes (incl. background)')


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ── YOLO-style single-cell prediction head (simplified) ──────────────
class YOLOHead(nn.Module):
    """
    Predict B boxes and C classes for each cell in the feature grid.
    Output shape: (B_batch, S, S, B*5 + C)
    where S = grid size, B = boxes per cell, C = num classes
    """
    def __init__(self, in_channels, S=7, B=2, C=20):
        super().__init__()
        self.S, self.B, self.C = S, B, C
        self.conv = nn.Conv2d(in_channels, B*5 + C, kernel_size=1)

    def forward(self, x):
        # x: (batch, in_channels, S, S)
        out = self.conv(x)                    # (batch, B*5+C, S, S)
        out = out.permute(0, 2, 3, 1)         # (batch, S, S, B*5+C)
        return out

# ── Focal Loss (RetinaNet) ────────────────────────────────────────────
class FocalLoss(nn.Module):
    """
    Focal loss = -(1-p_t)^gamma * log(p_t)
    Down-weights easy examples (high p_t) so hard examples dominate.
    gamma=0 reduces to standard cross-entropy.
    gamma=2 is the standard RetinaNet setting.
    """
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        # logits: (N, C), targets: (N,)
        ce_loss = F.cross_entropy(logits, targets, reduction='none')  # (N,)
        p_t     = torch.exp(-ce_loss)           # predicted probability of true class
        # Weight: (1 - p_t)^gamma
        # Easy examples: p_t ≈ 1 → weight ≈ 0 → ignored
        # Hard examples: p_t ≈ 0 → weight ≈ 1 → full loss
        focal_weight = (1 - p_t) ** self.gamma
        focal_loss   = self.alpha * focal_weight * ce_loss
        return focal_loss.mean()

# ── Demonstrate the class imbalance problem ───────────────────────────
focal = FocalLoss(alpha=0.25, gamma=2.0)
ce    = nn.CrossEntropyLoss()

# Simulate: 1 hard positive (model wrong) vs 99 easy negatives (model very right)
logits  = torch.cat([
    torch.tensor([[0.1, 0.9]]),    # hard positive: model predicts class 0, truth is 1
    torch.tensor([[10.0, -10.0]]).repeat(99, 1)  # easy negatives: model very confident
])
targets = torch.cat([torch.tensor([1]), torch.zeros(99, dtype=torch.long)])

ce_val    = ce(logits, targets)
focal_val = focal(logits, targets)
print(f'Cross-entropy loss: {ce_val:.4f}   (dominated by easy negatives)')
print(f'Focal loss:         {focal_val:.4f}  (hard positive dominates)')
# Focal loss is much smaller overall but the hard example contributes more

# Using pretrained RetinaNet from torchvision:
# from torchvision.models.detection import retinanet_resnet50_fpn
# model = retinanet_resnet50_fpn(weights='DEFAULT')


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

# ── FCN-style segmentation with skip connections ──────────────────────
class FCN8s(nn.Module):
    """
    FCN-8s: semantic segmentation via fully convolutional network
    with skip connections from /8 and /16 feature maps.
    Uses VGG16 backbone.
    """
    def __init__(self, num_classes=21):  # 21 = 20 PASCAL VOC + background
        super().__init__()
        vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)

        # Split VGG16 features into blocks by pooling layers
        # pool3 output: /8 of input resolution
        # pool4 output: /16 of input resolution
        # pool5 output: /32 of input resolution
        self.block1_3 = vgg.features[:17]   # up to and including pool3
        self.block4   = vgg.features[17:24] # pool4 block
        self.block5   = vgg.features[24:]   # pool5 block

        # 1×1 convolutions to produce per-class scores at each scale
        self.score_pool3 = nn.Conv2d(256, num_classes, 1)
        self.score_pool4 = nn.Conv2d(512, num_classes, 1)
        # Replace VGG FC layers with 1×1 convs (FCN conversion)
        self.score_pool5 = nn.Sequential(
            nn.Conv2d(512, 4096, 7, padding=3), nn.ReLU(inplace=True), nn.Dropout2d(),
            nn.Conv2d(4096, 4096, 1),            nn.ReLU(inplace=True), nn.Dropout2d(),
            nn.Conv2d(4096, num_classes, 1),
        )

    def forward(self, x):
        H, W = x.shape[2:]

        # Forward through each block, save intermediate outputs
        p3 = self.block1_3(x)    # /8
        p4 = self.block4(p3)     # /16
        p5 = self.block5(p4)     # /32

        # Class scores at each scale
        s5 = self.score_pool5(p5)              # /32 (coarse, semantic)
        s4 = self.score_pool4(p4)              # /16
        s3 = self.score_pool3(p3)              # /8  (fine, spatial)

        # Fuse: upsample coarser predictions and add to finer ones
        s5_up = F.interpolate(s5, size=s4.shape[2:], mode='bilinear',
                              align_corners=False)
        fused4 = s5_up + s4              # /16: semantics + some spatial

        f4_up  = F.interpolate(fused4, size=s3.shape[2:], mode='bilinear',
                               align_corners=False)
        fused3 = f4_up + s3              # /8:  semantics + more spatial

        # Upsample to full input resolution
        out = F.interpolate(fused3, size=(H, W), mode='bilinear',
                            align_corners=False)
        return out   # (B, num_classes, H, W) — per-pixel class logits

# ── Test ──────────────────────────────────────────────────────────────
fcn = FCN8s(num_classes=21)
x   = torch.randn(2, 3, 320, 320)
out = fcn(x)
print(f'Input:  {x.shape}')    # (2,  3, 320, 320)
print(f'Output: {out.shape}')  # (2, 21, 320, 320) — same H×W!

# Training: CrossEntropyLoss on per-pixel predictions
seg_mask = torch.randint(0, 21, (2, 320, 320))
loss = nn.CrossEntropyLoss(ignore_index=255)(out, seg_mask)  # 255 = ignore
print(f'Segmentation loss: {loss.item():.4f}')

# Use pretrained FCN from torchvision:
# from torchvision.models.segmentation import fcn_resnet50
# model = fcn_resnet50(weights='DEFAULT')  # COCO pre-trained


In [ ]:
import torch
import torch.nn as nn

# ── U-Net: symmetric encoder-decoder with concatenation skips ─────────
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)

class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, features=[64,128,256,512]):
        super().__init__()
        self.encoders = nn.ModuleList()
        self.pools    = nn.ModuleList()
        self.upconvs  = nn.ModuleList()
        self.decoders = nn.ModuleList()

        # ── Encoder (contracting path) ────────────────────────────────
        in_ch = in_channels
        for f in features:
            self.encoders.append(DoubleConv(in_ch, f))
            self.pools.append(nn.MaxPool2d(2, 2))
            in_ch = f

        # ── Bottleneck ────────────────────────────────────────────────
        self.bottleneck = DoubleConv(features[-1], features[-1] * 2)

        # ── Decoder (expanding path) ──────────────────────────────────
        for f in reversed(features):
            # Transposed conv: upsample from 2f channels to f
            self.upconvs.append(nn.ConvTranspose2d(f*2, f, kernel_size=2, stride=2))
            # After concat: f (upsampled) + f (skip) = 2f channels
            self.decoders.append(DoubleConv(f*2, f))

        self.head = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def forward(self, x):
        # ── Encode: save each encoder output as a skip connection ──────
        skips = []
        for enc, pool in zip(self.encoders, self.pools):
            x = enc(x)
            skips.append(x)   # <── save for skip connection
            x = pool(x)

        x = self.bottleneck(x)
        skips = skips[::-1]   # reverse: deepest encoder matches first decoder

        # ── Decode: upsample, CONCATENATE skip, double-conv ────────────
        for upconv, dec, skip in zip(self.upconvs, self.decoders, skips):
            x = upconv(x)                        # upsample
            x = torch.cat([skip, x], dim=1)      # CONCAT skip connection
            x = dec(x)                           # double conv

        return self.head(x)   # (B, out_channels, H, W)

# ── Test: output matches input spatial dimensions ─────────────────────
unet = UNet(in_channels=1, out_channels=1)    # grayscale → binary mask
x    = torch.randn(2, 1, 256, 256)
out  = unet(x)
print(f'Input:  {x.shape}')    # (2, 1, 256, 256)
print(f'Output: {out.shape}')  # (2, 1, 256, 256) — pixel-perfect same size

# Parameter count
n = sum(p.numel() for p in unet.parameters())
print(f'Parameters: {n/1e6:.1f}M')   # ≈ 31M

# ── Verify skip connections are doing something ───────────────────────
# Count decoder input channels: f_up + f_skip = 2f per stage
for i, (upconv, dec) in enumerate(zip(unet.upconvs, unet.decoders)):
    up_out = upconv.out_channels
    dec_in = dec.net[0].in_channels
    print(f'Decoder {i}: upconv→{up_out}ch  concat→{dec_in}ch  ({up_out}+{up_out}={dec_in})')
# Decoder 0: upconv→512ch  concat→1024ch (512+512=1024)
# Decoder 1: upconv→256ch  concat→ 512ch (256+256=512)  ...

# ── Training with binary cross-entropy (for binary segmentation) ──────
optimiser  = torch.optim.Adam(unet.parameters(), lr=1e-4)
bce_loss   = nn.BCEWithLogitsLoss()
mask_gt    = torch.randint(0, 2, (2, 1, 256, 256)).float()

optimiser.zero_grad()
pred = unet(x)
loss = bce_loss(pred, mask_gt)   # BCEWithLogitsLoss — no sigmoid needed
loss.backward()
optimiser.step()
print(f'Training loss: {loss.item():.4f}')

# For multi-class segmentation, use out_channels=num_classes
# and nn.CrossEntropyLoss() with integer target masks.
